# DINOv3 Embedding Analysis & Semantic Search
**NB 24** | Multi-city archive | DINOv3 SAT | FAISS semantic retrieval

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import json
import pathlib

import numpy as np

import pygeovision as pgv

client = pgv.PyGeoVision()
print(client)

from pygeovision.models.foundation.dinov3 import DINOv3Backbone, get_transform


In [ ]:
# Multi-city study areas for embedding comparison
STUDY_AREAS = {
    'London'    : (-0.15, 51.47, -0.10, 51.52),
    'Dubai'     : (55.25, 25.15, 55.35, 25.25),
    'Nairobi'   : (36.80, -1.30, 37.00, -1.10),
    'São Paulo' : (-46.70,-23.60,-46.50,-23.40),
    'Tokyo'     : (139.70, 35.67,139.80, 35.72),
}
PROVIDERS  = ['planetary_computer']
BANDS      = ['B02','B03','B04','B08']
DATA_DIR   = pathlib.Path('./results/nb24')
DATA_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# Search and download one scene per city
city_scenes = {}
for city, bbox in STUDY_AREAS.items():
    r  = client.search(bbox=bbox, date_range=('2024-04-01','2024-09-30'),
        providers=PROVIDERS, cloud_cover_max=15)
    if r:
        city_dir = DATA_DIR/city.lower().replace(' ','_')
        city_dir.mkdir(exist_ok=True)
        dl = client.download([r[0]], output_dir=str(city_dir), bands=BANDS,
            post_process=['reproject:EPSG:4326','cog'])
        if dl and dl[0].success:
            city_scenes[city] = dl[0].path
            print(f"  {city:<12}: {dl[0].path.name}  cloud={r[0].cloud_cover:.1f}%")


In [ ]:
# Validate and preprocess all cities with identical parameters
city_ready = {}
for city, scene in city_scenes.items():
    bbox = STUDY_AREAS[city]
    rep  = client.validator.validate(str(scene), required_bands=len(BANDS))
    out  = DATA_DIR / f'{city.lower().replace(" ","_")}_preprocessed.tif'
    r    = client.prepare_for_ai(str(scene), stack_bands=BANDS, bbox=bbox,
        normalise='scale_factor', scale_factor=10000.0,
        model_type='foundation', output_path=str(out))
    city_ready[city] = str(out)
    print(f"  {city:<12}: {r['steps']}  shape={r['shape']}")


In [ ]:
# Extract DINOv3 SAT embeddings for all preprocessed scenes
# ALWAYS use get_transform() — SAT model uses different normalisation than ImageNet
backbone  = DINOv3Backbone('dinov3_vitl16_sat')
transform = get_transform('dinov3_vitl16_sat')

embeddings = {}
for city, path in city_ready.items():
    if pathlib.Path(path).exists():
        emb = backbone.extract_embeddings(path)   # (1, 1024)
        embeddings[city] = emb.squeeze()
        print(f"  {city:<12}: embedding shape={emb.shape}  norm={np.linalg.norm(emb):.2f}")


In [ ]:
# Cosine similarity matrix between all city embeddings
import numpy as np

cities = list(embeddings.keys())
E      = np.array([embeddings[c] for c in cities])
norms  = np.linalg.norm(E, axis=1, keepdims=True)
E_norm = E / (norms + 1e-10)
sim_matrix = E_norm @ E_norm.T

print("\nCosine Similarity Matrix (DINOv3 SAT embeddings)")
print(f"{'':>14}" + "".join(f"{c:>12}" for c in cities))
print('-'*(14 + 12*len(cities)))
for i, city_i in enumerate(cities):
    row = f"{city_i:>14}" + "".join(f"{sim_matrix[i,j]:>12.3f}" for j in range(len(cities)))
    print(row)
print("\n1.000 = identical urban structure  |  <0.5 = very different")


In [ ]:
# FAISS-based semantic image search
try:
    import faiss
    d = list(embeddings.values())[0].shape[0]   # embedding dimension (1024)
    index = faiss.IndexFlatIP(d)                  # inner product = cosine similarity
    E_all = np.array([embeddings[c] for c in cities], dtype=np.float32)
    faiss.normalize_L2(E_all)
    index.add(E_all)
    print(f"FAISS index built: {index.ntotal} cities × {d}-dim embeddings")

    # Query: find city most similar to London
    query = E_all[cities.index('London')].reshape(1,-1)
    D, I  = index.search(query, k=len(cities))
    print("\nCities most similar to London (by DINOv3 SAT embedding):")
    for rank, (dist, idx) in enumerate(zip(D[0], I[0]), 1):
        print(f"  {rank}. {cities[idx]:<14}: similarity={dist:.3f}")
except ImportError:
    print("faiss-cpu not installed — pip install faiss-cpu for semantic search")
    print("Cosine similarities computed above are equivalent")


In [ ]:
# Save embeddings and index
embeddings_dict = {city: emb.tolist() for city, emb in embeddings.items()}
with open(DATA_DIR/'city_embeddings.json', 'w') as f:
    json.dump(embeddings_dict, f)
print(f"Embeddings saved: {DATA_DIR}/city_embeddings.json")
print(f"Cities: {list(embeddings.keys())}")


In [ ]:
# Export processed scenes as COG + generate validation reports
for scene_type, p in ready_paths.items():
    if pathlib.Path(p).exists():
        cog_out = DATA_DIR / f'{scene_type}_cog.tif'
        client.postprocess.to_cog(p, str(cog_out))
        rep = client.validator.validate(p)
        print(f'{scene_type:<18}: COG={cog_out.name}  valid={rep.passed}')
print(f'All {len(ready_paths)} preprocessed scenes exported as COG')


**NB24 — DINOv3 Embeddings** complete.
- Study area: 5 global cities
- Sensor: Sentinel-2 4-band
- Model: DINOv3 SAT-493M + FAISS
- Pipeline: search → download → validate_raw → preprocess → validate_clean → model → postprocess → COG